In [ ]:
import os
import pandas as pd

# ==========================================
# 1. 경로 설정 (정확한 상대 경로 지정)
# ==========================================
final_path = "final.csv"  # ./facamp2026/final.csv
results_path = os.path.join("..", "data", "results.csv")  # ./data/results.csv
output_path = "final_updated.csv"  # ./facamp2026/final_updated.csv 로 저장

# ==========================================
# 2. 데이터프레임 로드
# ==========================================
print("데이터를 로드하는 중입니다...")
if not os.path.exists(final_path):
    raise FileNotFoundError(
        f"'{final_path}'를 찾을 수 없습니다. 경로를 확인해 주세요."
    )
if not os.path.exists(results_path):
    raise FileNotFoundError(
        f"'{results_path}'를 찾을 수 없습니다. 경로를 확인해 주세요."
    )

df_final = pd.read_csv(final_path)

# results.csv에서 피처 다각화에 필요한 원본 컬럼들을 모두 가져옵니다.
df_results = pd.read_csv(results_path)[
    ["date", "home_team", "away_team", "tournament", "neutral", "country"]
]

# ==========================================
# 3. 중복 컬럼 방지 및 Merge
# ==========================================
# 새로 추가할 피처들이 기존에 있다면 중복 방지를 위해 미리 drop합니다.
new_cols = [
    "tournament",
    "tournament_weight",
    "is_neutral",
    "is_home_country_advantage",
    "is_world_cup",
    "is_friendly",
    "rank_diff_weight_inter",
]
cols_to_drop = [c for c in new_cols if c in df_final.columns]
if cols_to_drop:
    df_final = df_final.drop(columns=cols_to_drop)

print("결합(Merge)을 진행합니다...")
df_final = pd.merge(
    df_final, df_results, on=["date", "home_team", "away_team"], how="left"
)

# ==========================================
# 4. 대회 중요도 및 고도화 피처 엔지니어링 함수
# ==========================================


def get_tournament_weight(tournament):
    if pd.isna(tournament):
        return 1
    t_lower = tournament.lower()
    if "fifa world cup" in t_lower and "qualification" not in t_lower:
        return 4
    elif any(
        cup in t_lower
        for cup in [
            "copa américa",
            "uefa euro",
            "afcon",
            "asian cup",
            "conacaf gold cup",
        ]
    ):
        return 3
    elif "qualification" in t_lower or "nations league" in t_lower:
        return 2
    else:
        return 1


print("고도화된 신규 피처들을 생성하고 있습니다...")

# [신규 피처 1] 수치형 대회 가중치
df_final["tournament_weight"] = df_final["tournament"].apply(
    get_tournament_weight
)

# [신규 피처 2] 중립 경기장 여부 바이너리 변환 (True -> 1, False -> 0)
df_final["is_neutral"] = df_final["neutral"].astype(int)

# [신규 피처 3] 실질적 개최국 홈 어드밴티지 (실제 경기가 홈팀의 국가에서 열리는가?)
df_final["is_home_country_advantage"] = (
    df_final["home_team"] == df_final["country"]
).astype(int)

# [신규 피처 4] 대회 성격 원-핫 인코딩 (바이너리 플래그)
df_final["is_world_cup"] = df_final["tournament"].apply(
    lambda x: 1
    if pd.notna(x) and "fifa world cup" in x.lower() and "qualification" not in x.lower()
    else 0
)
df_final["is_friendly"] = df_final["tournament"].apply(
    lambda x: 1 if pd.notna(x) and "friendly" in x.lower() else 0
)

# [신규 피처 5] 인터랙션(상호작용) 피처: 랭킹 차이 * 대회 중요도
# 중요 대회의 진검승부일수록 상위 랭커와 하위 랭커의 격차가 결과에 절대적으로 반영되는 경향을 캐치합니다.
df_final["rank_diff_weight_inter"] = (
    df_final["rank_diff"] * df_final["tournament_weight"]
)

# ==========================================
# 5. 불필요한 원본 문자열 컬럼 제거 및 저장
# ==========================================
# 모델에 그대로 넣으면 에러가 나는 문자열 및 중복 불필요 컬럼들 drop
df_final = df_final.drop(columns=["tournament", "neutral", "country"])

# 기존 final.csv의 모든 컬럼을 보존한 채 신규 피처가 더해진 형태로 파일 저장
df_final.to_csv(output_path, index=False)

print("-" * 60)
print(f"기존 피처를 모두 보존하고 5개의 고차원 변수가 추가되었습니다!")
print(f"최종 저장 완료: {os.path.abspath(output_path)}")
print("-" * 60)

데이터를 로드하는 중입니다...
결합(Merge)을 진행합니다...
대회 가중치(tournament_weight) 피처를 생성하고 있습니다...
------------------------------------------------------------
성공적으로 새로운 피처가 추가된 파일이 생성되었습니다!
저장된 위치: /Users/seoyoun/Documents/facamp2026/final_updated.csv
------------------------------------------------------------
